# Qwen2.5-7B-Instruct Sales & Marketing QLoRA Fine-Tuning
Updated from the Gemma-2B notebook with key fixes for quality and stability:
1. **Clean Qwen2.5-7B-Instruct base** (no inherited banking checkpoint, full QLoRA `r=16` on attention + MLP layers).
2. **Multi-turn ChatML formatting (`messages`)** instead of isolated contextless turns + raw `### Instruction:` strings.
3. **Auto-detected `bf16` vs `fp16`** (prevents P100/T4 hardware mismatch).
4. **Validation split (`eval_loss`)**, `max_length=1024`, unified `SFTConfig`, and direct **Hugging Face Hub LoRA push** for ZeroGPU deployment.

In [ ]:
# ============================================
# CELL 1: Install Required Packages
# ============================================
!pip install -q -U bitsandbytes==0.49.1 peft==0.18.1 trl==0.27.0 accelerate==1.12.0 datasets==4.5.0 transformers==4.57.5 huggingface_hub
print("✅ Packages installed successfully!")

In [ ]:
# ============================================
# CELL 2: Imports & GPU Capability Detection
# ============================================
import os
import json
import shutil
import torch
import transformers
from datasets import load_dataset, Dataset
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, AutoPeftModelForCausalLM
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16

print(f"Transformers: {transformers.__version__}")
print(f"PyTorch:      {torch.__version__}")
print(f"CUDA:         {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:          {torch.cuda.get_device_name(0)}")
    print(f"VRAM (GB):    {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f}")
print(f"Precision:    {'bf16' if USE_BF16 else 'fp16 (T4/P100 safe)'}")

In [ ]:
# ============================================
# CELL 3: HuggingFace Authentication (Colab & Kaggle Compatible)
# ============================================
from huggingface_hub import login

hf_token = os.getenv("HF_TOKEN", "")
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN") or ""
    except Exception:
        pass

if not hf_token:
    try:
        from kaggle_secrets import UserSecretsClient
        hf_token = UserSecretsClient().get_secret("HF_TOKEN") or ""
    except Exception:
        pass

os.environ["HF_TOKEN"] = hf_token

if hf_token:
    login(token=hf_token)
    print("✅ Logged into HuggingFace Hub!")
else:
    print("⚠️ HF_TOKEN not found in Colab Secrets. Public Qwen2.5-7B downloads work without a token,")
    print("   but add HF_TOKEN in Colab's left-sidebar Secrets (🔑) if you want to push_to_hub.")

In [ ]:
# ============================================
# CELL 4: Configuration (Colab-Ready)
# ============================================
BASE_MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
WORK_DIR = "/content" if os.path.exists("/content") else "/kaggle/working"
OUTPUT_DIR = f"{WORK_DIR}/qwen2.5-7b-sales-checkpoints"
FINAL_ADAPTER_DIR = f"{WORK_DIR}/qwen2.5-7b-sales-lora-final"
HF_HUB_REPO_ID = "abdullahsallehaqeel123/bizcall-qwen7b-sales-lora"  # Change if needed

SYSTEM_PROMPT = (
    "You are an expert B2B outbound sales, marketing, and objection-handling AI assistant. "
    "Write persuasive, natural, concise, and conversion-focused responses tailored to the prospect's context."
)
print(f"Working directory: {WORK_DIR}")

In [ ]:
# ============================================
# CELL 5: Load & Format Multi-Turn Sales Dataset
# ============================================
print("⏳ Loading goendalf666/sales-conversations...")
sales_ds = load_dataset("goendalf666/sales-conversations", split="train")
print(f"✅ Raw conversations loaded: {len(sales_ds):,}")

def clean_turn(text: str, prefix: str) -> str:
    text = (text or "").strip()
    if text.startswith(prefix):
        text = text[len(prefix):].strip()
    return text

def build_multiturn_messages(example):
    """Preserve full Customer <-> Salesman conversation context in ChatML messages."""
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    valid_turns = 0

    for i in range(0, 20, 2):
        cust = example.get(str(i))
        sales = example.get(str(i + 1))
        if not cust or not sales:
            break
        cust_clean = clean_turn(cust, "Customer:")
        sales_clean = clean_turn(sales, "Salesman:")
        if len(cust_clean) < 2 or len(sales_clean) < 2:
            continue
        messages.append({"role": "user", "content": cust_clean})
        messages.append({"role": "assistant", "content": sales_clean})
        valid_turns += 1

    if valid_turns == 0:
        return None
    return {"messages": messages}

formatted_rows = []
for ex in sales_ds:
    item = build_multiturn_messages(ex)
    if item is not None:
        formatted_rows.append(item)

full_dataset = Dataset.from_list(formatted_rows).shuffle(seed=42)
split_ds = full_dataset.train_test_split(test_size=0.05, seed=42)
train_dataset = split_ds["train"]
eval_dataset = split_ds["test"]

print(f"✅ Clean multi-turn conversations: {len(full_dataset):,}")
print(f"   - Train samples: {len(train_dataset):,}")
print(f"   - Eval samples:  {len(eval_dataset):,}")
print(f"   - Sample turns:  {len(train_dataset[0]['messages']) - 1}")

In [ ]:
# ============================================
# CELL 6: Load Qwen2.5-7B-Instruct (4-bit QLoRA) + Tokenizer
# ============================================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=COMPUTE_DTYPE,
    trust_remote_code=True,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

print(f"✅ Loaded {BASE_MODEL_ID} on {model.device}")

In [ ]:
# ============================================
# CELL 7: Attach Fresh QLoRA Adapters
# ============================================
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# ============================================
# CELL 8: Configure SFTConfig & Initialize SFTTrainer
# ============================================
vram_gb = (torch.cuda.get_device_properties(0).total_memory / 1e9) if torch.cuda.is_available() else 0
if vram_gb >= 35:  # A100 (40GB / 80GB)
    BATCH_SIZE, GRAD_ACCUM = 4, 2
elif vram_gb >= 22:  # L4 (24GB)
    BATCH_SIZE, GRAD_ACCUM = 2, 4
else:  # T4 / P100 (16GB)
    BATCH_SIZE, GRAD_ACCUM = 1, 8

print(f"Auto-selected batch={BATCH_SIZE}, grad_accum={GRAD_ACCUM} (effective batch={BATCH_SIZE * GRAD_ACCUM}) for {vram_gb:.1f} GB VRAM")

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    max_length=1024,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=2,
    learning_rate=2e-4,             # Standard QLoRA LR for fresh adapter
    fp16=not USE_BF16,
    bf16=USE_BF16,
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    max_grad_norm=0.3,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to="none",
    push_to_hub=False,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

print("✅ SFTTrainer initialized for Qwen2.5-7B-Instruct!")

In [ ]:
# ============================================
# CELL 9: Train
# ============================================
print("🚀 Starting Qwen2.5-7B QLoRA training...")
trainer.train()
print("✅ Training complete!")

In [ ]:
# ============================================
# CELL 10: Save LoRA Adapter Locally + Zip + Optional Hub Push
# ============================================
trainer.model.save_pretrained(FINAL_ADAPTER_DIR)
tokenizer.save_pretrained(FINAL_ADAPTER_DIR)
print(f"✅ Saved LoRA adapter weights to {FINAL_ADAPTER_DIR}")

zip_path = shutil.make_archive(f"{WORK_DIR}/qwen2.5-7b-sales-lora", "zip", FINAL_ADAPTER_DIR)
size_mb = os.path.getsize(zip_path) / (1024 * 1024)
print(f"📦 Zipped LoRA adapter: {zip_path} ({size_mb:.2f} MB)")

# Optional: download zip straight to your PC from Colab:
# from google.colab import files
# files.download(zip_path)

# Optional: push ONLY the LoRA adapter (~160 MB) directly to HuggingFace Hub for ZeroGPU:
# if os.environ.get("HF_TOKEN"):
#     trainer.model.push_to_hub(HF_HUB_REPO_ID, private=True)
#     tokenizer.push_to_hub(HF_HUB_REPO_ID, private=True)
#     print(f"🚀 Pushed LoRA adapter to https://huggingface.co/{HF_HUB_REPO_ID}")

In [ ]:
# ============================================
# CELL 11: Inference Verification (Single-Call ChatML)
# ============================================
import gc

del trainer, model
gc.collect()
torch.cuda.empty_cache()

eval_model = AutoPeftModelForCausalLM.from_pretrained(
    FINAL_ADAPTER_DIR,
    quantization_config=bnb_config,
    torch_dtype=COMPUTE_DTYPE,
    device_map={"": 0},
)
eval_model.eval()
eval_tokenizer = AutoTokenizer.from_pretrained(FINAL_ADAPTER_DIR)

def generate_sales_reply(user_prompt: str, history=None, max_new_tokens: int = 350, temperature: float = 0.7):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    if history:
        messages.extend(history)
    messages.append({"role": "user", "content": user_prompt})

    prompt_text = eval_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = eval_tokenizer(prompt_text, return_tensors="pt").to(eval_model.device)

    with torch.no_grad():
        outputs = eval_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.08,
            eos_token_id=eval_tokenizer.eos_token_id,
            pad_token_id=eval_tokenizer.pad_token_id,
        )

    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
    return eval_tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

test_prompts = [
    "Create a 3-sentence high-converting pitch for our AI voice call center BizCall AI to a dental clinic.",
    "Your product is too expensive compared to competitors.",
    "Write a follow-up email for a B2B prospect who requested pricing for BizCall AI but went silent.",
]
for p in test_prompts:
    print("=" * 80)
    print("PROMPT:", p)
    print("REPLY:\n", generate_sales_reply(p))